# AutoGluon.TimeSeries V1.4 Cheatsheet

This notebook contains the key code snippets from the AutoGluon Time Series cheatsheet.

In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

/Users/sergio/code/automl_data_science/.env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from process_data import *

data_dir = os.path.join(os.getcwd(), 'data')

generacion = pd.read_csv(os.path.join(data_dir,'generacion.csv'))
generacion['timestamp'] = pd.to_datetime(generacion['timestamp'])

In [3]:
generacion.columns

Index(['Codigo Central', 'timestamp', 'Generacion_MWh'], dtype='object')

In [4]:
grouped_generacion=generacion.groupby("Codigo Central")

In [5]:
grouped_generacion.count()

,timestamp,Generacion_MWh
Codigo Central,,
Canela,8377,8377
Canela 2,8377,8377
El Arrayán,4992,4992
EÓLICA VALLE DE LOS VIENTOS_Eólico,7272,7272
Eólica Los Cururos,4488,4488
Eólica Negrete,3576,3576
Eólica Taltal,1728,1728
Eólica Totoral,8377,8377
HUAYCA1,7295,7295


In [6]:
min_date=generacion['timestamp'].min()
max_date=generacion['timestamp'].max()

In [7]:
min_date,max_date

(Timestamp('2014-01-01 00:00:00'), Timestamp('2014-12-31 23:00:00'))

In [8]:
full_date_range = pd.date_range(start=min_date, end=max_date, freq='h')

In [9]:
len(full_date_range)

8760

In [10]:
reindexed_dfs = []
for station,df in grouped_generacion:
    df.set_index('timestamp',inplace=True)
    df=df[~df.index.duplicated(keep='first')]
    df_reindexed = df.reindex(full_date_range)
    df_reindexed['Codigo Central']=df_reindexed['Codigo Central'].bfill()
    df_reindexed['Codigo Central']=df_reindexed['Codigo Central'].ffill()
    df_reindexed['Generacion_MWh']=df_reindexed['Generacion_MWh'].bfill()    
    df_reindexed['Generacion_MWh']=df_reindexed['Generacion_MWh'].interpolate()    
    reindexed_dfs.append(df_reindexed)
    
    #df_reindexed = df.reindex(full_date_range)

In [11]:
generacion_reindexed=pd.concat(reindexed_dfs)

In [23]:
generacion_reindexed.groupby('Codigo Central').count()

,timestamp,Generacion_MWh
Codigo Central,,
Canela,8760,8760
Canela 2,8760,8760
El Arrayán,8760,8760
EÓLICA VALLE DE LOS VIENTOS_Eólico,8760,8760
Eólica Los Cururos,8760,8760
Eólica Negrete,8760,8760
Eólica Taltal,8760,8760
Eólica Totoral,8760,8760
HUAYCA1,8760,8760


In [13]:
generacion_reindexed.reset_index(names='timestamp',inplace=True)

In [14]:
generacion_reindexed

,timestamp,Codigo Central,Generacion_MWh
0,2014-01-01 00:00:00,Canela,4.30
1,2014-01-01 01:00:00,Canela,2.30
2,2014-01-01 02:00:00,Canela,2.40
3,2014-01-01 03:00:00,Canela,1.60
4,2014-01-01 04:00:00,Canela,1.50
...,...,...,...
315355,2014-12-31 19:00:00,Ucuquer 2,0.94
315356,2014-12-31 20:00:00,Ucuquer 2,0.94
315357,2014-12-31 21:00:00,Ucuquer 2,0.94
315358,2014-12-31 22:00:00,Ucuquer 2,0.94


### Convert Raw Data into a TimeSeriesDataFrame

Convert your raw data into the format required by AutoGluon: `TimeSeriesDataFrame`.

In [15]:
def train_test_split(data,split_fraction,feature_keys):
    data=data[feature_keys]
    train_split = int(split_fraction * int(data.shape[0]))
    data=data._get_numeric_data()
    data_mean = data[:train_split].mean(axis=0)
    data_std = data[:train_split].std(axis=0)
    data = (data - data_mean) / data_std
    train_data = data.iloc[0 : train_split - 1]
    val_data = data.iloc[train_split:]
    return train_data,val_data


def create_batch(data,lag,future):
    df_lag=pd.concat([data[:-future].shift(i) for i in range(lag-1,-1,-1)],axis=1).dropna()
    #df_lag.columns=['pm_'+str(i) for i in range(lag,-1,1)]
    X=df_lag.values
    y=data[future+lag-1:].values
    return X,y

def create_batch_multistep(df,lag,future,feature=None):
    if feature is None:
        data=df
    else:
        data=df[feature]
    data.fillna(0.0,inplace=True)
    df_lag=pd.concat([data[:-future].shift(i) for i in range(lag-1,-1,-1)],axis=1).dropna()
    df_future=pd.concat([data[lag-1:].shift(-i) for i in range(1,future+1)],axis=1).dropna()
    X=df_lag.values
    y=df_future.values
    return X,y

In [16]:
import numpy as np 

split_fraction = 0.8
feature_keys = ['Generacion_MWh']
X_train_datasets=list()
y_train_datasets=list()
X_test_datasets=list()
y_test_datasets=list()
future=24*7
past=2*24*7

dataset_names=list()
for item_id, gdf in generacion_reindexed.groupby('Codigo Central'):
    train,test=train_test_split(gdf,split_fraction,feature_keys)
    train_multi_feature=list()
    test_multi_feature=list()
    for feature in train.columns:
        X_train,y_train=create_batch_multistep(train,past,future,feature)
        X_test,y_test=create_batch_multistep(test,past,future,feature)
        train_multi_feature.append(X_train)
        test_multi_feature.append(X_test)
        if feature=='Generacion_MWh':
            y_train_datasets.append(y_train)
            y_test_datasets.append(y_test)
    X_train_datasets.append(np.stack(train_multi_feature,axis=-1))
    X_test_datasets.append(np.stack(test_multi_feature,axis=-1))
    dataset_names.append(item_id)
X_train_datasets=np.stack(X_train_datasets,axis=0)
y_train_datasets=np.stack(y_train_datasets,axis=0)
X_test_datasets=np.stack(X_test_datasets,axis=0)
y_test_datasets=np.stack(y_test_datasets,axis=0)

In [30]:
X_train_datasets.shape,X_test_datasets.shape

((36, 6504, 336, 1), (36, 1249, 336, 1))

In [31]:
y_train_datasets.shape

(36, 6504, 168)

# Train Global Model

In [104]:
import torch
import torch.nn as nn

class LSTMForecast(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2, dropout=0.2):
        super(LSTMForecast, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc_mean = nn.Linear(hidden_size, output_size)
        self.fc_var = nn.Linear(hidden_size, output_size)
        self.softplus = nn.Softplus(beta=1.0, threshold=20.0)
        
    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]        
        mean = self.fc_mean(x)  # Take output from the last LSTM cell
        var = self.softplus(self.fc_var(x)) 
        return mean,var

In [114]:
X_train_global=X_train_datasets.reshape(36*6504,336,1)
X_test_global=X_test_datasets.reshape(36*1249,336,1)
y_train_global=y_train_datasets.reshape(36*6504,168)
y_test_global=y_test_datasets.reshape(36*1249,168)

In [135]:
X_test_global.shape

(44964, 336, 1)

In [115]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS device for acceleration.")
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print("MPS device not available, falling back to CPU.")


print(f"Using device: {device}")

Using MPS device for acceleration.
Using device: mps


In [119]:
def get_dataloader(X,y,batch_size,axis=0,device='cpu'):
    num_train=X.shape[axis]
    indices = np.array(list(range(0,num_train)))
    indices=np.random.permutation(indices)
    for i in range(0, len(indices),batch_size):
        batch_indices = indices[i: i+batch_size]
        X_batch=torch.from_numpy(X[batch_indices,:,:].astype(np.float32)).to(device)
        y_batch=torch.from_numpy(y[batch_indices,:].astype(np.float32)).to(device)
        yield X_batch,y_batch 
              

In [120]:
from torch import optim 

model=LSTMForecast(input_size=1,hidden_size=32,output_size=168).to(device)
loss_fn = nn.GaussianNLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
n_epochs = 100
batch_size=128
loss_history=list()
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in get_dataloader(X_train_global,y_train_global,batch_size,0,device):
        y_pred,y_var = model(X_batch)
        loss = loss_fn(y_batch, y_pred,y_var)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # Validation
    loss_history.append(loss.item())
    if epoch % 10 != 0:
        continue
    model.eval()
    with torch.no_grad():
        for X_batch, y_batch in get_dataloader(X_test_global,y_test_global,batch_size,0,device):
            y_pred,y_var = model(X_batch)
            test_loss = loss_fn(y_batch, y_pred,y_var)
            break
    print("Epoch %d: train RMSE %.4f, test RMSE %.4f" % (epoch, loss_history[-1], test_loss.item()))

Epoch 0: train RMSE -0.6631, test RMSE inf
Epoch 10: train RMSE -1.0660, test RMSE inf
